## 강화학습(Reinforcement Learning)
- 기계학습 중 하나
- 어떤 환경을 탐색하는 에이전트가 현재 상태를 인식하여 행동을 취함으로써 보상을 얻는다.
- 에이전트가 앞으로 누적될 보상을 최대화하는 일련의 행동으로 정의되는 정책을 찾는다.

### Reward와 Delayed Reward
- **보상**(Reward)
    - 에이전트가 행동을 취한 후 환경으로부터 받는 피드백
- **지연된 보상**(Delayed Reward)
    - 행동과 보상 사이에 시간 지연이 있거나 이후 행동과 합쳐져 더 좋은 보상을 얻을 수 있는 경우
    - 이 경우 그동안의 행동 중 어떤 행동이 보상에 기여했는지 판단이 어려움

## Markov Decision Process (MDP)
- 1800년대 러시아 수학자 Andrey Markov가 제안한 확률 과정
- 시스템이 현재 상태에만 의존하고 과거 상태에는 의존하지 않는다는 가정
- 강화학습에서 MDP는 에이전트가 환경과 상호작용하는 모델로 사용
- MDP는 다음과 같은 요소로 구성:
1. **상태 집합**(State Space, $S$): 에이전트가 처할 수 있는 모든 가능한 상태들의 집합
2. **행동 집합**(Action Space, $A$): 에이전트가 취할 수 있는 모든 가능한 행동들의 집합
3. **전이 확률 행렬**(Transition Probability Matrix, $\mathcal{P}$): 특정 상태에서 특정 행동을 취했을 때 다음 상태로 전이될 확률
4. **보상 함수**(Reward Function, $r$): 특정 상태에서 특정 행동을 취했을 때 받는 보상을 정의하는 함수
5. **할인 인자**(Discount Factor, $\gamma$): 미래 보상의 현재 가치를 결정하는 요소. $0 \leq \gamma \leq 1$

--- 
다음과 같은 게임화면(그리드 월드)이 있다고 하자.

|(1, 1)|(2,1)|(3,1)|(4,1)|(5,1)|
|---|---|---|---|---|
|(1, 2)|(2,2)|(3,2)|(4,2)|(5,2)|
|(1, 3)|(2,3)|(3,3)|(4,3)|(5,3)|
|(1, 4)|(2,4)|(3,4)|(4,4)|(5,4)|
|(1, 5)|(2,5)|(3,5)|(4,5)|(5,5)|

- $S = \{(1,1), (2,1), \cdots, (5,5)\}$: 에이전트가 위치할 수 있는 모든 상태
- 시간 $t$에서
    - $S_t = s$: 상태
    - $A_t = a$: 에이전트가 취할 행동 (예: $\text{상}$, $\text{하}$, $\text{좌}$, $\text{우}$)
- 하지만 $A_t$가 $s$에서 $s'$로 이동하는 행동이 항상 성공하는 것은 아니다. 예를 들어, $s = (1,1)$에서 $a = \text{하}$을 취하면 $s' = (1,2)$로 이동할 확률이 $0.8$이고, 실패하여 $s' = (1,1)$에 머무를 확률이 $0.2$일 수 있다.
    - deterministic 이 아닌 stochastic하다.
- 이는 전이 확률 $\mathcal{P}$에 의해 표현된다.
    - $\mathcal{P}_{ss'}^{a} = \mathbb{P}[S_{t+1}=s'|S_t=s, A_t=a]$ *($\mathbb{P}$는 조건부 확률)*
    - 위의 상황에 대하여 $\mathcal{P}_{(1,1)(1,2)}^{하} = 0.8$
- 에이전트가 행동을 취하면 그에 따른 보상을 환경이 에이전트에게 알려주며 에이전트는 관찰(observation)로 변한 상태를 알게 된다.
    - 이때 보상 $r$은 $r(s, a) = \mathbb{E}[R_{t+1}|S_t = s, A_t = a]$ *($\mathbb{E}$는 기댓값)*
    - 강화학습에서는 정답이나 사전 지식 없이 이 보상을 통해 학습한다.
    - 이 보상을 immediate reward라고 하는데, 에이전트는 즉각적으로 나오는 보상 뿐 아니라 이후 얻는 보상까지 고려한다.
- 현재 보상이 이후 보상보다 더 선호된다.
    - 방법 - 보상의 가치를 기하급수적으로 감소시킨다.
        - $A_t$ 에 대한 보상 가치: $R_{t+1} = 1$
        - 다음 스텝 보상 가치: $\gamma R_{t+2}$
        - 두 스텝 뒤 보상 가치: $\gamma^2 R_{t+3}$
        - 누적 보상의 합: $G_t = \sum_{i=1}^{\infty}{\gamma ^ {i-1} R_{t+i}}$
    - ex. 평지(-1) -> 평지(-1) -> 목적지(+100) 순으로 보상을 받을 때 $\gamma = 0.9$ 라면
        - $G_t = (-1) + 0.9 \times (-1) + 0.81 \times 100 = -1 -0.9 +81 = 79.1$

### 에이전트
- 에이전트는 아래 요소를 하나 이상 가지게 된다.
1) **정책** Policy($\pi$): 에이전트의 행동 함수
    - state에 대한 action을 반환한다.
    - deterministic policy: $\pi(a|s) = a$
    - stochastic policy: $\pi = \mathbb{P}[A_t=a|S_t=s]$
2) **가치 함수** Value function($\text{v}$): 각 상태나 행동에 대한 보상($\mathbb{E}[G_t$])을 계산하는 함수
    - 에피소드: 시작과 끝이 있는 에이전트와 환경의 상호작용 단위(ex. 게임시작-게임오버).
        - 에피소드에서는 에피소드를 끝낼 마지막 상태가 있다.
    - 가치 함수 $\text{v}(s) = \mathbb{E}[G_t|S_t=s]$
        - $G_t = \sum_{i=1}^{\infty}{\gamma ^ {i-1} R_{t+i}} = R_{t+1} + \gamma \cdot R_{t+2} + \gamma ^ 2 \cdot R_{t+3} \cdots = R_{t+1} + \gamma \cdot (R_{t+2} + \gamma \cdot R_{t+3} \cdots) = R_{t+1} + \gamma \cdot G(t+1)$
        - $\therefore \text{v}(s) = \mathbb{E}[R_{t+1} + \gamma \cdot G(t+1)|S_t=s] = \mathbb{E}[R_{t+1} + \gamma \cdot v(S_{t+1})|S_t=s]$
    - 정책을 고려한 가치 함수 $\text{v}_{\pi}(s) = \mathbb{E}_{\pi}[R_{t+1} + \gamma \cdot v(S_{t+1})|S_t=s]$
3) **모델** Model: 에이전트의 환경에 대한 예측기. 전이 확률이나 보상을 예측한다.

### Q Function(Quality Function)
- **상태 가치 함수**(State Value Function)
    - 상태가 입력으로 들어오면 그 상태에서 앞으로 받을 보상의 합을 반환하는 함수
- **행동 가치 함수**(Action Value Function, Q-Function)
    - 특정 상태에서 특정 행동을 취했을 때 그 상태와 행동을 시작으로 받을 보상의 합을 반환하는 함수
    - 이때 가치 함수 $\text{v}_{pi} (s) = \sum_{a \in A} {\pi (A|s) \cdot q_{\pi}(s, a)}$
        - 즉 가치 함수는 q함수의 평균(기댓값)
    - 이때 q함수 $q_{\pi} (s, a) = \mathbb{E}_{\pi} [R_{t+1} + \gamma \cdot q_{\pi} (S_{t+1}, A_{t+1})|S_t=s,A_t=a]$ 로 정의됨.
    - 강화학습에서는 에이전트가 행동을 선택하는 기준으로 가치함수보다 q함수를 더 사용함.